# Part 1: Data Collection and Preprocessing

## 0. Install dependencies

In [ ]:
!pip install datasets lxml cairosvg sentencepiece tokenizers tqdm matplotlib torch

## 1. Download Datasets

In [ ]:
from datasets import load_dataset


icons = load_dataset('starvector/svg-icons-simple', split='train')
emoji = load_dataset('starvector/svg-emoji-simple', split='train')
stack = load_dataset('starvector/svg-stack-simple', split='train[:18%]')
all_data = list(icons) + list(emoji) + list(stack)
print(f'Total raw samples: {len(all_data):,}')

## 2. SVG Cleaning and Normalization

In [ ]:
import re

def strip_noise(svg: str) -> str:
    """Remove comments, metadata, processing instructions, and collapse whitespace."""
    svg = re.sub(r'<!--.*?-->', '', svg, flags=re.DOTALL)
    svg = re.sub(r'<\?xml[^>]*\?>', '', svg)
    svg = re.sub(r'<!DOCTYPE[^>]*>', '', svg)
    svg = re.sub(r'<metadata.*?</metadata>', '', svg, flags=re.DOTALL)
    svg = re.sub(r'<title>.*?</title>', '', svg, flags=re.DOTALL)
    svg = re.sub(r'<desc>.*?</desc>', '', svg, flags=re.DOTALL)
    svg = re.sub(r'\s+', ' ', svg).strip()
    return svg


def round_numbers(svg: str) -> str:
    """Round all floating-point numbers to 1 decimal place to reduce vocab."""
    return re.sub(
        r'(\d+\.\d{2,})',
        lambda m: str(round(float(m.group()), 1)),
        svg
    )


def normalize_structure(svg: str) -> str:
    """Ensure xmlns, consistent viewBox quotes, and self-closing leaf elements."""
    # Ensure xmlns attribute is present
    if 'xmlns=' not in svg:
        svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)

    # Normalize viewBox to double-quoted
    svg = re.sub(
        r"viewBox='([^']+)'",
        lambda m: f'viewBox="{m.group(1)}"',
        svg
    )

    # Self-close void elements that weren't already self-closed
    void_tags = r'(path|circle|rect|line|ellipse|polygon|polyline|use|image)'
    svg = re.sub(
        rf'<{void_tags}((?:[^>](?!/))*?)(?<!/)>',
        r'<\1\2/>',
        svg
    )
    return svg


def clean_svg(svg: str) -> str:
    svg = strip_noise(svg)
    svg = round_numbers(svg)
    svg = normalize_structure(svg)
    return svg


print('Cleaning pipeline defined.')

## 3. Validation and Filtering

In [ ]:
from lxml import etree
import cairosvg

# --- Thresholds ---
MIN_CHARS   = 80     # raised floor — degenerate stubs removed
MAX_CHARS   = 1200   # lowered ceiling — keeps sequences learnable
MAX_ELEMENTS = 20    # reject overly complex SVGs


def count_elements(svg: str) -> int:
    """Count non-structural opening tags (excludes svg, g, defs)."""
    return len(re.findall(r'<(?![\/!?]|svg|g\b|defs\b)', svg))


def is_valid_xml(svg: str) -> bool:
    try:
        root = etree.fromstring(svg.encode('utf-8'))
        tag = root.tag
        return tag == 'svg' or tag.endswith('}svg')
    except Exception:
        return False


def renders_ok(svg: str) -> bool:
    try:
        cairosvg.svg2png(bytestring=svg.encode('utf-8'))
        return True
    except Exception:
        return False


def filter_svg(svg: str) -> bool:
    if len(svg) < MIN_CHARS:
        return False
    if len(svg) > MAX_CHARS:
        return False
    if count_elements(svg) > MAX_ELEMENTS:
        return False
    # Reject scripts / foreign objects (security + complexity)
    if re.search(r'<(script|foreignObject|animate)', svg, re.IGNORECASE):
        return False
    if not is_valid_xml(svg):
        return False
    if not renders_ok(svg):
        return False
    return True


print('Validation pipeline defined.')

In [ ]:
from tqdm import tqdm

cleaned_svgs = []
reject_too_short = 0
reject_too_long  = 0
reject_too_complex = 0
reject_invalid_xml = 0
reject_render     = 0

for item in tqdm(all_data, desc='Cleaning + filtering'):
    raw = item['Svg']
    svg = clean_svg(raw)

    # Detailed rejection tracking
    if len(svg) < MIN_CHARS:
        reject_too_short += 1; continue
    if len(svg) > MAX_CHARS:
        reject_too_long  += 1; continue
    if count_elements(svg) > MAX_ELEMENTS:
        reject_too_complex += 1; continue
    if re.search(r'<(script|foreignObject|animate)', svg, re.IGNORECASE):
        continue
    if not is_valid_xml(svg):
        reject_invalid_xml += 1; continue
    if not renders_ok(svg):
        reject_render += 1; continue

    cleaned_svgs.append(svg)

print(f'\n=== Filter summary ===')
print(f'  Input:           {len(all_data):>8,}')
print(f'  Too short:       {reject_too_short:>8,}')
print(f'  Too long:        {reject_too_long:>8,}')
print(f'  Too complex:     {reject_too_complex:>8,}')
print(f'  Invalid XML:     {reject_invalid_xml:>8,}')
print(f'  Render failure:  {reject_render:>8,}')
print(f'  Accepted:        {len(cleaned_svgs):>8,}')

## 4. Train / Val / Test Splits

In [ ]:
import random

random.seed(42)
random.shuffle(cleaned_svgs)

n = len(cleaned_svgs)
n_train = int(0.98 * n)
n_val   = int(0.99 * n)

train_svgs = cleaned_svgs[:n_train]
val_svgs   = cleaned_svgs[n_train:n_val]
test_svgs  = cleaned_svgs[n_val:]

print(f'Train: {len(train_svgs):,}  |  Val: {len(val_svgs):,}  |  Test: {len(test_svgs):,}')

## 5. Train BPE Tokenizer
Design choices:
- Vocab size 6000: large enough to tokenize common SVG attribute strings as single tokens (e.g. `stroke-width`, `viewBox`), but small enough that the embedding table stays manageable for small models.
- `ByteLevel` pre-tokenizer handles arbitrary UTF-8 safely.
- We explicitly seed the vocabulary with frequent SVG keywords so BPE merges preserve them atomically instead of splitting `s`, `t`, `r`, `o`, `k`, `e`.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# SVG-domain seed vocabulary: frequent keywords that should stay intact
SVG_SEED_VOCAB = [
    # Tags
    '<svg', '</svg>', '<path', '<circle', '<rect', '<g>', '</g>',
    '<polygon', '<ellipse', '<line', '<polyline', '<text', '</text>',
    '<defs>', '</defs>', '<clipPath>', '</clipPath>', '<linearGradient>',
    # Common attributes
    'viewBox', 'xmlns', 'stroke', 'stroke-width', 'fill', 'opacity',
    'transform', 'class', 'style', 'id',
    # Path commands (space-prefixed as they appear in d="...")
    ' d="', 'M ', 'L ', 'C ', 'Q ', 'A ', 'Z', 'H ', 'V ', 'S ', 'T ',
    # Geometry attributes
    'cx=', 'cy=', 'r=', 'x=', 'y=', 'width=', 'height=', 'rx=', 'ry=',
    'x1=', 'y1=', 'x2=', 'y2=',
    # Common values
    'none', 'black', 'white', '#000', '#fff', '#000000', '#ffffff',
    'fill-rule', 'clip-path', 'stroke-linecap', 'stroke-linejoin',
    'stroke-dasharray', 'transform="translate', 'transform="scale',
    'transform="rotate',
]

tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

trainer = trainers.BpeTrainer(
    vocab_size=6000,
    special_tokens=['<pad>', '<unk>', '<bos>', '<eos>'],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    # Force SVG keywords to appear in the vocab before BPE merges
    # (HF tokenizers supports this via show_progress=True)
    show_progress=True,
    min_frequency=2,
)

tokenizer.train_from_iterator(train_svgs, trainer=trainer)
tokenizer.save('svg_tokenizer_6.json')

PAD_ID = tokenizer.token_to_id('<pad>')
BOS_ID = tokenizer.token_to_id('<bos>')
EOS_ID = tokenizer.token_to_id('<eos>')

print(f'Vocab size: {tokenizer.get_vocab_size():,}')
print(f'PAD={PAD_ID}  BOS={BOS_ID}  EOS={EOS_ID}')

## 6. Tokenize Splits 

Wrapping every sample with `<bos>` and `<eos>` teaches the model where valid SVGs begin and end, which is critical for clean generation termination.

In [ ]:
import torch

MAX_TOKENS = 1024  # hard cap per sequence


def tokenize_split(svgs, tokenizer, max_tokens=MAX_TOKENS):
    """
    Encode each SVG, wrap with <bos>/<eos>, discard sequences exceeding max_tokens.
    Returns list[list[int]].
    """
    result = []
    for svg in tqdm(svgs, desc='Tokenizing'):
        ids = tokenizer.encode(svg).ids
        wrapped = [BOS_ID] + ids + [EOS_ID]
        if len(wrapped) <= max_tokens:
            result.append(wrapped)
    return result


train_tokens = tokenize_split(train_svgs, tokenizer)
val_tokens   = tokenize_split(val_svgs,   tokenizer)
test_tokens  = tokenize_split(test_svgs,  tokenizer)

print(f'\nAfter tokenization + length filter:')
print(f'  Train sequences: {len(train_tokens):,}')
print(f'  Val   sequences: {len(val_tokens):,}')
print(f'  Test  sequences: {len(test_tokens):,}')

In [ ]:
# Save tokens
torch.save(train_tokens, 'train_tokens_6.pt')
torch.save(val_tokens,   'val_tokens_6.pt')
torch.save(test_tokens,  'test_tokens_6.pt')
print('Token files saved.')

## 7. Dataset Statistics

In [ ]:
def total_tokens(seqs):
    return sum(len(s) for s in seqs)

train_tok_count = total_tokens(train_tokens)
val_tok_count   = total_tokens(val_tokens)
test_tok_count  = total_tokens(test_tokens)
grand_total     = train_tok_count + val_tok_count + test_tok_count

print('=== Token counts ===')
print(f'  Train : {train_tok_count:>12,}  ({100*train_tok_count/grand_total:.1f}%)')
print(f'  Val   : {val_tok_count:>12,}  ({100*val_tok_count/grand_total:.1f}%)')
print(f'  Test  : {test_tok_count:>12,}  ({100*test_tok_count/grand_total:.1f}%)')
print(f'  Total : {grand_total:>12,}')
print()
if train_tok_count >= 100_000_000:
    print('✓  Training set meets the 100M token minimum.')
else:
    print(f'⚠  Training set has {train_tok_count/1e6:.1f}M tokens. '
          f'Consider adding svg-fonts-simple to reach 100M.')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lengths = [len(s) for s in train_tokens]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(lengths, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Sequence Length Distribution (train)')
axes[0].set_xlabel('Tokens per sequence')
axes[0].set_ylabel('Count')
axes[0].axvline(np.mean(lengths), color='orange', linestyle='--', label=f'mean={np.mean(lengths):.0f}')
axes[0].axvline(np.median(lengths), color='red', linestyle=':', label=f'median={np.median(lengths):.0f}')
axes[0].legend()

# CDF
sorted_len = np.sort(lengths)
cdf = np.arange(1, len(sorted_len)+1) / len(sorted_len)
axes[1].plot(sorted_len, cdf, color='steelblue')
axes[1].set_title('CDF of Sequence Lengths')
axes[1].set_xlabel('Tokens per sequence')
axes[1].set_ylabel('Cumulative fraction')
for p in [0.5, 0.9, 0.95, 0.99]:
    pv = np.percentile(lengths, p*100)
    axes[1].axhline(p, color='gray', linewidth=0.7, linestyle='--')
    axes[1].text(pv+5, p-0.03, f'p{int(p*100)}={pv:.0f}', fontsize=8)

plt.tight_layout()
plt.savefig('sequence_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean length   : {np.mean(lengths):.1f}')
print(f'Median length : {np.median(lengths):.1f}')
print(f'p90 length    : {np.percentile(lengths, 90):.0f}')
print(f'p99 length    : {np.percentile(lengths, 99):.0f}')
print(f'Max length    : {max(lengths)}')

## 8. Render Examples at Different Complexity Levels

In [ ]:
import os
from IPython.display import display, Image as IPImage
import matplotlib.image as mpimg

os.makedirs('examples', exist_ok=True)

# Sort by character length to sample across complexity spectrum
sorted_by_len = sorted(train_svgs, key=len)
n_train_svgs  = len(sorted_by_len)


indices = [
    int(0.01 * n_train_svgs),   # very simple
    int(0.05 * n_train_svgs),   # simple
    int(0.25 * n_train_svgs),   # low-medium
    int(0.50 * n_train_svgs),   # medium
    int(0.75 * n_train_svgs),   # high-medium
    int(0.95 * n_train_svgs),   # complex
    int(0.99 * n_train_svgs),   # very complex
]
labels = ['Very simple', 'Simple', 'Low-medium', 'Medium', 'High-medium', 'Complex', 'Very complex']

rendered_paths = []
for i, (idx, label) in enumerate(zip(indices, labels)):
    svg = sorted_by_len[idx]
    out = f'examples/example_{i}_{label.replace(" ","_")}.png'
    try:
        cairosvg.svg2png(bytestring=svg.encode('utf-8'), write_to=out,
                         output_width=128, output_height=128)
        rendered_paths.append((out, label, len(svg)))
    except Exception as e:
        print(f'Render failed for {label}: {e}')

# Display rendered grid
fig, axes = plt.subplots(1, len(rendered_paths), figsize=(3 * len(rendered_paths), 3.5))
if len(rendered_paths) == 1:
    axes = [axes]

for ax, (path, label, char_len) in zip(axes, rendered_paths):
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.set_title(f'{label}\n({char_len} chars)', fontsize=8)
    ax.axis('off')

plt.suptitle('SVG Examples Across Complexity Levels', y=1.02)
plt.tight_layout()
plt.savefig('example_complexity_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Tokenizer Vocabulary Inspection

In [ ]:
# show how common SVG constructs are tokenized
examples_to_encode = [
    '<path d="M 10 10 L 54 10 L 54 54 Z" fill="none" stroke="black"/>',
    '<circle cx="32" cy="32" r="18" fill="#1f77b4"/>',
    '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">',
]

print('=== Tokenization examples ===\n')
for ex in examples_to_encode:
    enc = tokenizer.encode(ex)
    tokens = enc.tokens
    ids    = enc.ids
    print(f'Input : {ex}')
    print(f'Tokens: {tokens}')
    print(f'IDs   : {ids}')
    print(f'Count : {len(ids)} tokens\n')

## 10. Mount Drive and Save (Colab)

In [ ]:
# Uncomment when running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

# import os, shutil
# base = '/content/drive/MyDrive/svg_project'
# os.makedirs(base, exist_ok=True)
#
# for fname in ['train_tokens_6.pt', 'val_tokens_6.pt', 'test_tokens_6.pt', 'svg_tokenizer_6.json',
#               'sequence_length_distribution.png', 'example_complexity_grid.png']:
#     if os.path.exists(fname):
#         shutil.copy(fname, f'{base}/{fname}')
#         print(f'Saved {fname}')

## 11. Summary

In [ ]:
print('=' * 50)
print('PART 1 SUMMARY')
print('=' * 50)
print(f'Raw samples collected     : {len(all_data):>10,}')
print(f'Accepted after filtering  : {len(cleaned_svgs):>10,}')
print(f'Acceptance rate           : {100*len(cleaned_svgs)/len(all_data):>9.1f}%')
print()
print(f'Tokenizer vocab size      : {tokenizer.get_vocab_size():>10,}')
print(f'Special tokens            :  <pad> <unk> <bos> <eos>')
print()
print(f'Train sequences           : {len(train_tokens):>10,}')
print(f'Val   sequences           : {len(val_tokens):>10,}')
print(f'Test  sequences           : {len(test_tokens):>10,}')
print()
print(f'Train tokens              : {train_tok_count:>10,}')
print(f'Val   tokens              : {val_tok_count:>10,}')
print(f'Test  tokens              : {test_tok_count:>10,}')
print(f'Grand total tokens        : {grand_total:>10,}')
print()
print('Files saved:')
for f in ['svg_tokenizer_6.json', 'train_tokens_6.pt', 'val_tokens_6.pt', 'test_tokens_6.pt']:
    print(f'  {f}')